# Stage 2: Task Definition & Formatting
Loads raw Parquet splits and formats inputs/targets into standard task formats (e.g., Morphological Inflection). Saves as Hugging Face Datasets on disk.

In [1]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    RAW_DIR = '/content/drive/MyDrive/lora/data/raw-splits'
    PROCESSED_DIR = '/content/drive/MyDrive/lora/data/df-task'
except ImportError:
    RAW_DIR = '../data/raw-splits'
    PROCESSED_DIR = '../data/df-task'

import os
import pandas as pd
from datasets import Dataset, DatasetDict

os.makedirs(PROCESSED_DIR, exist_ok=True)

LANGUAGES = ['eng', 'spa', 'swe']  # ISO 639-3 codes for UniMorph languages

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
def format_task_variants(df):

    # Standard prefix format
    df['input_v1_prefix'] = 'lemma: ' + df['lemma'] + ' | features: ' + df['features']
    
    # Feature-first format (Puts context tags closer to generation start)
    df['input_v2_features_first'] = df['features'] + ' ' + df['lemma']
    
    # Clean bracketed / tagged format (Common in SigMorphon shared tasks)
    df['input_v3_sigmorphon'] = '<' + df['features'].str.replace(';', '><') + '> ' + df['lemma']
    
    # Natural language prompt (Best for general LLMs / instruction-tuned models)
    df['input_v4_instruction'] = 'Inflect the lemma "' + df['lemma'] + '" with features: ' + df['features']
    
    df['target_text'] = df['target']

    return df

In [3]:
def build_task_dataset(lang):
    lang_raw_dir = os.path.join(RAW_DIR, lang)
    if not os.path.exists(lang_raw_dir):
        print(f'Raw data for {lang} not found.')
        return
    
    ds_dict = {}
    for split in ['train', 'val', 'test']:
        parquet_path = os.path.join(lang_raw_dir, f'{split}.parquet')
        df = pd.read_parquet(parquet_path)
        formatted_df = format_task_variants(df)
        ds_dict[split] = Dataset.from_pandas(formatted_df)
    
    dataset = DatasetDict(ds_dict)
    output_path = os.path.join(PROCESSED_DIR, f'{lang}_inflection')
    dataset.save_to_disk(output_path)
    print(f'Saved task dataset for {lang} to {output_path}')

for lang in LANGUAGES:
    build_task_dataset(lang)

Saving the dataset (0/1 shards):   0%|          | 0/521740 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/65222 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/65506 [00:00<?, ? examples/s]

Saved task dataset for eng to /content/drive/MyDrive/unimorph_exp/data/df-task/eng_inflection


Saving the dataset (0/1 shards):   0%|          | 0/953806 [00:00<?, ? examples/s]

...

Saved task dataset for spa to /content/drive/MyDrive/unimorph_exp/data/df-task/spa_inflection


Saving the dataset (0/1 shards):   0%|          | 0/105154 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/13413 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/13130 [00:00<?, ? examples/s]

Saved task dataset for swe to /content/drive/MyDrive/unimorph_exp/data/df-task/swe_inflection
